# Test Smell - Single‑Agent Notebook

This Colab notebook performs **test smell detection and refactoring** using a **single‑agent workflow**.

**How it works**

* **Dependencies & runtime** – The first code cells install all required Python libraries and start an Ollama server directly in Colab (or connect to an existing one via `base_url`).
* **Parameter widgets** – Provide the CSV containing the test cases, the path to the smell‑definition file, select the smell to analyse, the Ollama `model` (e.g. `phi4`, `llama3`), sampling `temperature`, server `base_url`.
* **Main loop** – For each test case, a single agent is prompted with the smell definition and the code snippet. It decides whether the smell is present and, if so, produces a refactored version that preserves the original behaviour. The output is appended directly to the input CSV, ensuring all changes are traceable and ready for evaluation.

In [ ]:
# @title Install dependencies
%pip uninstall -q -y langchain langchain-core langchain-community langchain-ollama langchain-text-splitters
%pip install -q --no-deps lightrag[ollama]
%pip install -q "langchain==0.3.*" "langchain-core==0.3.*" "langchain-community==0.3.*" "langchain-ollama==0.3.*"

In [ ]:
# @title Install and start Ollama runtime
!sudo apt-get -qq update && sudo apt-get -y -qq install pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os, threading, subprocess, time
def start_ollama():
    os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
    os.environ["OLLAMA_ORIGINS"] = "*"
    subprocess.Popen(["ollama", "serve"])
threading.Thread(target=start_ollama, daemon=True).start()
time.sleep(5)
print("=== verificação ===")
!ollama --version

In [ ]:
# @title Create parameter widgets
import ipywidgets as widgets
from IPython.display import display

csv_path_widget = widgets.Text(value="Dataset.csv", description='CSV:')
definitions_path_widget = widgets.Text(value="test_smell_definitions_and_refactorings.txt", description='Defs:')
smell = "Conditional Test Logic" # @param ["Assertion Roulette", "Magic Number Test", "Duplicate Assert", "Exception Handling", "Conditional Test Logic"]
smell_widget = widgets.Text(value=smell, description='Smell:')
model_widget = widgets.Text(value="gemma4:31b", description='Model:')
temperature_widget = widgets.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.1, description='Temp:')
base_url_widget = widgets.Text(value="http://localhost:11434", description='Base URL:')

display(csv_path_widget, definitions_path_widget, model_widget,
        temperature_widget, base_url_widget)

Text(value='Dataset.csv', description='CSV:')

Text(value='test_smell_definitions_and_refactorings.txt', description='Defs:')

Text(value='gemma4:31b', description='Model:')

FloatSlider(value=0.6, description='Temp:', max=1.0)

Text(value='http://localhost:11434', description='Base URL:')

In [ ]:
# @title Capture parameters from widgets
csv_path = csv_path_widget.value
definitions_path = definitions_path_widget.value
smell = smell_widget.value
model = model_widget.value
temperature = temperature_widget.value
base_url = base_url_widget.value

In [ ]:
# @title Define prompt templates and helpers
from __future__ import annotations

import argparse
import os
import re
import shutil
import subprocess
import sys
import threading
import time
from datetime import date
from pathlib import Path

import pandas as pd
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

def load_smell(def_text: str, smell_name: str) -> tuple[str, str]:
    pat = re.compile(
        r'test_smell_name\s*=\s*"([^"]+)"\s*[\r\n]+'
        r'test_smell_definition\s*=\s*"""(.*?)"""',
        re.S,
    )
    for name, definition in pat.findall(def_text):
        if name.strip().lower() == smell_name.lower():
            return name.strip(), definition.strip()
    raise ValueError(f'Didnt find Smell "{smell_name}".')

In [ ]:
!pip install tqdm
from tqdm.notebook import tqdm

In [ ]:
!ollama --version
!curl -s http://localhost:11434/api/tags && echo " <- servidor OK"

In [ ]:
# @title Run detection and evaluation loop

code = "{code}"

def build_prompts(test_smell_name: str, test_smell_definition_and_refactoring: str) -> PromptTemplate:
    return PromptTemplate(
        input_variables=["code"],
        template=f"""
You are a coding assistant specializing in test code analysis and refactoring, with many years of experience.
{test_smell_definition_and_refactoring}

Your task:
Analyze the provided test code to identify and resolve occurrences of "{test_smell_name}".
If no such smell is present, return the original code unchanged. Ensure that the refactored test maintains the same behavior while eliminating the {test_smell_name}.
Finally, output only the final refactored code, ensuring it is valid under JUnit 5 and free of compilation errors, without providing any additional explanations or text.

Code to analyze:
{code}
"""
    )

with open(definitions_path, "r", encoding="utf-8") as f:
    definitions_text = f.read()

smell_name, smell_def = load_smell(definitions_text, smell)

!ollama pull {model}
prompt = build_prompts(smell_name, smell_def)
llm = OllamaLLM(model=model, base_url=base_url, temperature=temperature)
chain = LLMChain(llm=llm, prompt=prompt)

df = pd.read_csv(csv_path)

today = date.today().strftime("%Y-%m-%d")
for col in ["LLM", "Test Smell", "Date", "Refactored code"]:
    if col not in df.columns:
        df[col] = ""

with open("agentes.txt", "w", encoding="utf-8") as log:
  original_stdout = sys.stdout
  sys.stdout = log
  try:
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing tests"):
        code = str(row.get("Test Code", "")).strip()
        df.loc[idx, ["LLM", "Test Smell", "Date"]] = [model, smell_name, today]
        refactored = chain.run(code=code)
        df.at[idx, "Refactored code"] = refactored

        print(f"\n===== Test {idx+1} =====\n")
        print(f"Test Code: \n {code}")
        print(f"{idx + 1}. Result: {refactored}\n===========\n")
  finally:
    sys.stdout = original_stdout


output_csv = f"/content/output_{smell.replace(' ', '_')}.csv"
df.to_csv(output_csv, index=False)
print(f"Results saved to: {output_csv}")
print("Agent log written to agentes.txt")

In [ ]:
from google.colab import files

files.download(output_csv)
files.download("agentes.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>